# 04 · Bring your own graph: custom edge list, GEARS adapter, AnnData round-trip

**Problem.** Researchers rarely want only the built-in priors. They have a correlation network, a
hand-curated pathway, or a GEARS-format co-expression graph — and they live in scanpy/AnnData.
`graph-perturb` is designed so all of these drop straight in.

**Approach.** Three demos:
1. **Custom edge list.** Build a gene-gene CSV from a data-driven correlation prior, plug it in via
   `get_graph_source("custom", path=...)` (the `CustomEdgeListGraph` adapter), train, and compare to
   a built-in backend over the same split.
2. **BONUS — GEARS adapter.** Write a GEARS-style co-expression edge list and load it with
   `load_gears_graph` (and the `gears` registry backend).
3. **BONUS — AnnData round-trip.** Show `processed_to_anndata` / `anndata_to_processed` so the
   pipeline drops into an existing scanpy workflow and back out losslessly.

**What to look at.** That a custom graph is a first-class backend (same `PerturbationGraph`, same
training/eval path), the metrics table comparing it to GO, and the assertion that the AnnData
round-trip is exact.

In [ ]:
import logging
import tempfile
from pathlib import Path
import numpy as np
import pandas as pd
logging.basicConfig(level=logging.WARNING)
np.random.seed(0)

from graph_perturb.config import DataConfig, SplitConfig, ModelConfig, TrainConfig, EvalConfig
from graph_perturb.data import make_splits, anndata_to_processed, processed_to_anndata
from graph_perturb.data.norman import load_norman, make_synthetic_norman
from graph_perturb.data.dataset import build_dataloaders
from graph_perturb.graphs.registry import get_graph_source
from graph_perturb.graphs.gears_adapter import load_gears_graph
from graph_perturb.models import build_model
from graph_perturb.train import train_model
from graph_perturb.evaluate import evaluate_model, compare_backends

workdir = Path(tempfile.mkdtemp(prefix="gp_cookbook_"))
print("scratch dir:", workdir)

In [ ]:
try:
    data = load_norman(DataConfig(name="norman", n_top_genes=2000))
    MODE = "REAL Norman Perturb-seq"
except Exception as exc:
    print(f"[fallback] real Norman load failed ({type(exc).__name__}: {exc})")
    data = make_synthetic_norman(n_genes=80, n_conditions=30, seed=0)
    MODE = "SYNTHETIC offline stand-in"
print(f"DATA MODE: {MODE}  |  cells={data.n_cells} genes={data.n_genes}")

splits = make_splits(data, SplitConfig(test_single_frac=0.3, test_combo_frac=0.3, seed=0))
print({k: len(v) for k, v in splits.as_dict().items()})

## 1. Build a CUSTOM edge list from a correlation prior

We compute a gene-gene Pearson correlation over the (control + perturbed) expression matrix and keep
each gene's top-k strongest partners as edges, with the |correlation| as an edge weight. This is a
common, fully data-driven prior. We write it to a CSV with `source,target,weight` columns — exactly
the schema `CustomEdgeListGraph` expects.

In [ ]:
X = np.asarray(data.expression, dtype=np.float64)
corr = np.corrcoef(X.T)                 # [n_genes, n_genes]
corr = np.nan_to_num(corr)
np.fill_diagonal(corr, 0.0)
genes = list(data.gene_names)

K = 5
rows = []
for i, g in enumerate(genes):
    partners = np.argsort(np.abs(corr[i]))[-K:][::-1]
    for j in partners:
        if i < j:  # undirected: emit each pair once
            rows.append((g, genes[j], float(abs(corr[i, j]))))
edge_df = pd.DataFrame(rows, columns=["source", "target", "weight"]).drop_duplicates(["source", "target"])
custom_csv = workdir / "corr_prior_edges.csv"
edge_df.to_csv(custom_csv, index=False)
print(f"wrote {len(edge_df)} custom edges -> {custom_csv}")
edge_df.head()

## 2. Plug the custom graph in via `get_graph_source("custom", options=...)`

The custom backend is registered under the key `"custom"`; backend-specific options (`path`,
`source_col`, `target_col`, `weight_col`) are passed straight through. The result is an ordinary
`PerturbationGraph`, indistinguishable to the rest of the pipeline from GO/Reactome/STRING.

In [ ]:
custom_source = get_graph_source(
    "custom",
    path=str(custom_csv),
    source_col="source",
    target_col="target",
    weight_col="weight",
)
custom_graph = custom_source.build(data.gene_names, use_cache=False)
print(custom_graph)
print("weighted:", custom_graph.edge_weight is not None)

## 3. Train on the custom graph and compare to a built-in backend (GO)

Same model, same split — only the graph prior differs.

In [ ]:
model_cfg = ModelConfig(name="gnn", hidden_dim=32, n_layers=2, dropout=0.1, attention_heads=2)
train_cfg = TrainConfig(epochs=5, batch_size=16, lr=1e-3, device="cpu",
                        early_stop_patience=5, log_every=100, seed=0)
eval_cfg = EvalConfig(overlap_k=20, splits=("test_single", "test_combo"))

graphs = {
    "custom_corr": custom_graph,
    "go_bp": get_graph_source("go_bp").build(data.gene_names, use_cache=False),
}

results_by_backend = {}
for label, g in graphs.items():
    print(f"=== {label} ===")
    loaders = build_dataloaders(data, g, splits, train_cfg)
    model = build_model(model_cfg, num_genes=data.n_genes, graph=g)
    train_model(model, loaders, train_cfg, ckpt_dir=None)
    results_by_backend[label] = evaluate_model(model, data, g, splits, eval_cfg)

print("\ncustom-vs-builtin:")
print(compare_backends(results_by_backend).to_string())

## 4. BONUS — GEARS-format adapter

[GEARS](https://github.com/snap-stanford/GEARS) drives its GNN with a gene co-expression / GO graph,
shipped as a networkx pickle, a co-expression edge list, or a PyG bundle. The `gears` backend reads
all three. Here we write a small GEARS-style edge list (`gene1,gene2,importance`) and load it two
ways: the convenience wrapper `load_gears_graph` (derives its own gene universe) and the registered
`gears` backend (aligned to our gene universe).

In [ ]:
gears_df = edge_df.rename(columns={"source": "gene1", "target": "gene2", "weight": "importance"})
gears_csv = workdir / "gears_coexpr.csv"
gears_df.to_csv(gears_csv, index=False)

# (a) convenience wrapper: self-contained universe from the file's own nodes
gears_graph_self = load_gears_graph(str(gears_csv), use_cache=False)
print("load_gears_graph (own universe):", gears_graph_self)

# (b) registered backend aligned to OUR gene universe (drops into the pipeline)
gears_source = get_graph_source("gears", path=str(gears_csv), format="edgelist")
gears_graph = gears_source.build(data.gene_names, use_cache=False)
print("gears backend (our universe):", gears_graph)
assert list(gears_graph.gene_names) == list(data.gene_names)

## 5. BONUS — AnnData round-trip for scanpy workflows

`processed_to_anndata` exports our processed container to a standard `AnnData` (expression in `.X`,
canonical labels in `obs['condition']`, baseline stashed in `uns`). `anndata_to_processed` reverses
it exactly — so users can preprocess in scanpy, hand us the `AnnData`, and recover the same data.

In [ ]:
adata = processed_to_anndata(data)
print(adata)
print("obs['condition'] head:", list(adata.obs['condition'][:5]))

roundtrip = anndata_to_processed(adata, condition_key="condition", control_label="ctrl")

assert list(roundtrip.gene_names) == list(data.gene_names)
assert np.array_equal(roundtrip.conditions, data.conditions)
assert np.allclose(roundtrip.expression, data.expression)
assert np.allclose(roundtrip.control_mean, data.control_mean)
print("AnnData round-trip is EXACT (expression, genes, conditions, baseline all match).")

**Takeaway.** A user-supplied correlation network, a GEARS file, and a scanpy `AnnData` are all
first-class citizens: the custom edge list trains through the identical pipeline as the built-in
priors, the GEARS adapter normalizes external graph formats into the same `PerturbationGraph`, and
the lossless AnnData round-trip means `graph-perturb` slots into existing single-cell workflows
without forcing users to adopt our loader.